# 01 — Sanity checks

This is the correctness gate for the Yamada benchmarks. It runs the published and independent checks before any performance claim is accepted.

The checks include published tree/cycle/bouquet/theta formulas, the bridge and one-point-union identities, the published planar $K_4$ value, direct-vs-recursive Negami evaluation, both public Yamada backends, the one-crossing theta/mirror relation, and projection invariance of the normalized Yamada polynomial across many valid generic views of fixed trivalent embeddings.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'knotted_graph').exists():
    raise RuntimeError('Run this notebook from inside the KnottedGraph checkout.')

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f'A stale knotted_graph was imported from {kg_path}')

env = dict(os.environ)
env['PYTHONPATH'] = str(SRC)
env['PYTHONNOUSERSITE'] = '1'
script = ROOT / 'dev' / 'run_yamada_sanity_checks.py'
proc = subprocess.run([sys.executable, str(script)], cwd=ROOT, env=env, text=True, capture_output=True)
print(proc.stdout)
if proc.returncode:
    raise RuntimeError(f'Sanity checks failed.\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}')
assert 'PASS: all published/independent Yamada sanity checks succeeded.' in proc.stdout
print('branch-local KnottedGraph:', kg_path)


## Projection-invariance sanity check for trivalent embeddings

The public graph entry point normally samples several projections and keeps the one with the fewest crossings. That policy is useful for speed, but by itself it does not verify that the complete

$$
G_{3D}\longrightarrow\text{projection}\longrightarrow\text{PD code}\longrightarrow\Upsilon
$$

pipeline respects projection invariance. The check below therefore fixes one spatial embedding at a time, samples many valid generic projections, computes the **normalized** Yamada polynomial for every sampled projection, and requires every result to agree.

The test is deliberately restricted to trivalent $\theta$-graphs (two degree-3 vertices). Different projections may have different crossing counts and different PD codes; those diagrammatic changes are allowed. The invariant itself must remain unchanged. Invalid or nongeneric sampled views are already rejected by `sample_projections(...)` and are not counted as successful projections.


In [ ]:
import networkx as nx
import numpy as np
import sympy as sp

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
)

A = sp.Symbol('A')
PROJECTION_SAMPLES = 24
MIN_VALID_PROJECTIONS = 12


def _same_polynomial(left, right):
    return sp.simplify(sp.together(sp.expand(left - right))) == 0


def _embedded_one_crossing_theta(*, mirror=False):
    zsign = -1.0 if mirror else 1.0
    graph = nx.MultiGraph()
    graph.add_node('u', pos=np.array([-2.0, 0.0, 0.0]))
    graph.add_node('v', pos=np.array([2.0, 0.0, 0.0]))
    curves = [
        np.array([[-2, 0, 0], [-1, -1, 0.5 * zsign], [1, 1, 0.5 * zsign], [2, 0, 0]], dtype=float),
        np.array([[-2, 0, 0], [-1, 1, -0.5 * zsign], [1, -1, -0.5 * zsign], [2, 0, 0]], dtype=float),
        np.array([[-2, 0, 0], [-1, 2, 0], [1, 2, 0], [2, 0, 0]], dtype=float),
    ]
    for pts in curves:
        graph.add_edge('u', 'v', pts=pts)
    return graph


def _check_projection_invariance(label, graph):
    degrees = sorted(dict(graph.degree()).values())
    assert graph.number_of_nodes() == 2
    assert graph.number_of_edges() == 3
    assert degrees == [3, 3], f'{label}: benchmark graph is not trivalent: {degrees}'

    projections = sample_projections(
        graph,
        num_rotation_samples=PROJECTION_SAMPLES,
    )
    if len(projections) < MIN_VALID_PROJECTIONS:
        raise AssertionError(
            f'{label}: only {len(projections)}/{PROJECTION_SAMPLES} sampled views were valid; '
            f'expected at least {MIN_VALID_PROJECTIONS}.'
        )

    records = []
    for projection in projections:
        result = compute_yamada_polynomial(
            graph,
            A,
            rotation_angles=projection.rotation_angles,
            rotation_order=projection.rotation_order,
            crossing_warning_threshold=None,
            normalize=True,
            n_jobs=1,
            method='negami',
            return_result=True,
        )
        assert result.projection.pd_code == projection.pd_code
        assert result.projection.num_crossings == projection.num_crossings
        records.append((projection, sp.expand(result.polynomial)))

    reference = records[0][1]
    mismatches = [
        (index, projection.rotation_angles, projection.num_crossings, polynomial)
        for index, (projection, polynomial) in enumerate(records)
        if not _same_polynomial(polynomial, reference)
    ]
    if mismatches:
        details = '\n'.join(
            f'  sample={index}, angles={angles}, crossings={crossings}, Yamada={polynomial}'
            for index, angles, crossings, polynomial in mismatches[:5]
        )
        raise AssertionError(
            f'{label}: normalized Yamada changed across projections.\n'
            f'reference={reference}\n{details}'
        )

    crossing_counts = [projection.num_crossings for projection, _ in records]
    distinct_pd_codes = len({projection.pd_code for projection, _ in records})
    print(
        f'PASS  {label}: {len(records)}/{PROJECTION_SAMPLES} valid projections, '
        f'crossings={min(crossing_counts)}..{max(crossing_counts)}, '
        f'distinct PD codes={distinct_pd_codes}, distinct Yamada values=1'
    )
    print(f'      normalized Yamada = {reference}')
    return records


projection_invariance_records = {
    'theta': _check_projection_invariance(
        'trivalent one-crossing theta',
        _embedded_one_crossing_theta(),
    ),
    'theta_mirror': _check_projection_invariance(
        'trivalent one-crossing theta mirror',
        _embedded_one_crossing_theta(mirror=True),
    ),
}

print('PASS: normalized Yamada is projection-invariant for every valid sampled trivalent diagram.')


### Interpretation

A pass here is stronger than the ordinary projection-selection policy: the test does **not** compute $\Upsilon$ only for the easiest diagram. Instead, every valid sampled diagram of each fixed trivalent embedding is evaluated. Crossing counts and PD codes may vary, but the normalized polynomial must have exactly one value within each embedding. This simultaneously exercises rotation, crossing detection, over/under assignment, arc construction, PD encoding, and Yamada evaluation.


## Acceptance criterion

Every published/independent assertion above must pass, and every valid sampled projection of each fixed trivalent benchmark embedding must produce the same normalized Yamada polynomial. This notebook contains no speed claim; the following notebooks are only meaningful after this one is green.
